In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import heapq

# ==========================================
# [필수 설정] OS에 맞는 한글 폰트 설정 (Windows는 맑은 고딕, Mac은 AppleGothic)
# ==========================================
plt.rc('font', family='Malgun Gothic') # Windows 사용자의 경우
# plt.rc('font', family='AppleGothic') # Mac 사용자의 경우
plt.rcParams['axes.unicode_minus'] = False

# 1) 모든 음식점을 그래프로 그려라 (데이터는 코드 안에 포함)
G = nx.Graph()
edges = [
    ("트라토리아 진", "다온카츠", 160),
    ("다온카츠", "익선취향", 120),
    ("익선취향", "마복림", 40),
    ("마복림", "빠오즈푸", 55),
    ("마복림", "땀땀", 110),
    ("빠오즈푸", "당금", 95),
    ("땀땀", "당금", 160),
    ("땀땀", "이쿠", 130)
]
G.add_weighted_edges_from(edges)

# 노드 위치 고정 (매 프레임마다 노드가 움직이지 않도록)
pos = nx.spring_layout(G, seed=42)

# --- 애니메이션 프레임 그리기 함수 ---
def draw_graph_frame(visited_nodes, visited_edges, current_node, ax, title):
    ax.clear()
    ax.set_title(title, fontsize=15)
    
    # 기본 그래프 (회색)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='lightgray', node_size=800)
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='lightgray', width=2)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=10, font_family=plt.rcParams['font.family'])
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax, font_size=9)
    
    # 방문한 노드와 간선 강조 (파란색 계열)
    if visited_nodes:
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=visited_nodes, node_color='skyblue', node_size=800)
    if visited_edges:
        nx.draw_networkx_edges(G, pos, ax=ax, edgelist=visited_edges, edge_color='royalblue', width=3)
    # 현재 탐색 중인 노드 강조 (빨간색)
    if current_node:
        nx.draw_networkx_nodes(G, pos, ax=ax, nodelist=[current_node], node_color='salmon', node_size=800)

# --- 영상 저장 공통 함수 ---
def save_animation(frames, filename, title):
    print(f"[{filename}] 영상 생성 중...")
    fig, ax = plt.subplots(figsize=(10, 7))
    
    def update(frame_idx):
        v_nodes, v_edges, curr = frames[frame_idx]
        draw_graph_frame(v_nodes, v_edges, curr, ax, title)
        
    ani = FuncAnimation(fig, update, frames=len(frames), interval=1000, repeat=False)
    # fps=1은 1초에 1프레임씩 진행됨을 의미
    ani.save(filename, writer='ffmpeg', fps=1)
    plt.close(fig)
    print(f"[{filename}] 저장 완료!\n")

# --- 알고리즘 구현 및 프레임 수집 ---
start_node = "마복림" # 임의의 시작점 설정

# 2) DFS 구현 및 프레임 수집
def get_dfs_frames(start):
    frames = []
    visited_nodes = []
    visited_edges = []
    
    def dfs(u, p=None):
        visited_nodes.append(u)
        if p:
            visited_edges.append((p, u))
        frames.append((list(visited_nodes), list(visited_edges), u))
        for v in G.neighbors(u):
            if v not in visited_nodes:
                dfs(v, u)
                
    dfs(start)
    return frames

# 3) BFS 구현 및 프레임 수집
def get_bfs_frames(start):
    frames = []
    visited_nodes = [start]
    visited_edges = []
    queue = [start]
    frames.append((list(visited_nodes), list(visited_edges), start))
    
    while queue:
        u = queue.pop(0)
        for v in G.neighbors(u):
            if v not in visited_nodes:
                visited_nodes.append(v)
                visited_edges.append((u, v))
                queue.append(v)
                frames.append((list(visited_nodes), list(visited_edges), v))
    return frames

# 4) 프림(Prim) 알고리즘 구현 및 프레임 수집
def get_prim_frames(start):
    frames = []
    visited_nodes = {start}
    mst_edges = []
    edges_pq = []
    
    for v, data in G[start].items():
        heapq.heappush(edges_pq, (data['weight'], start, v))
        
    frames.append((list(visited_nodes), list(mst_edges), start))
    
    while edges_pq and len(visited_nodes) < len(G.nodes):
        weight, u, v = heapq.heappop(edges_pq)
        if v not in visited_nodes:
            visited_nodes.add(v)
            mst_edges.append((u, v))
            frames.append((list(visited_nodes), list(mst_edges), v))
            
            for next_v, data in G[v].items():
                if next_v not in visited_nodes:
                    heapq.heappush(edges_pq, (data['weight'], v, next_v))
    return frames

# 5) 크루스칼(Kruskal) 알고리즘 구현 및 프레임 수집
def get_kruskal_frames():
    frames = []
    parent = {n: n for n in G.nodes}
    
    def find(i):
        if parent[i] == i: return i
        parent[i] = find(parent[i])
        return parent[i]
    
    def union(i, j):
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_i] = root_j

    sorted_edges = sorted(G.edges(data=True), key=lambda x: x[2]['weight'])
    mst_edges = []
    mst_nodes = set()
    
    frames.append((list(mst_nodes), list(mst_edges), None))
    for u, v, data in sorted_edges:
        if find(u) != find(v):
            union(u, v)
            mst_edges.append((u, v))
            mst_nodes.add(u)
            mst_nodes.add(v)
            # 크루스칼은 특정 탐색 노드가 없으므로 current_node는 None 처리
            frames.append((list(mst_nodes), list(mst_edges), None))
    return frames

# ==========================================
# [실행부] 4개의 애니메이션 파일 생성
# ==========================================
if __name__ == "__main__":
    print("애니메이션 생성을 시작합니다. 시간이 조금 걸릴 수 있습니다.\n")
    
    save_animation(get_dfs_frames(start_node), "dfs.mp4", "DFS 방문 과정")
    save_animation(get_bfs_frames(start_node), "bfs.mp4", "BFS 방문 과정")
    save_animation(get_prim_frames(start_node), "prim.mp4", "Prim 알고리즘 (최소 시간)")
    save_animation(get_kruskal_frames(), "kruskal.mp4", "Kruskal 알고리즘 (최소 시간)")
    
    print("모든 영상 생성이 완료되었습니다!")